In [ ]:
# The code in this block comes directly from data_sort_and_split.ipynb
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

heart_data = pd.read_csv("heart.csv")


heart_data['Sex_F'] = heart_data['Sex'].map({'M': 0, 'F': 1})
heart_data['ExerciseAngina'] = heart_data['ExerciseAngina'].map({'N': 0, 'Y': 1})

heart_data = heart_data.drop(columns=['Sex'])

heart_data['ChestPainType'] = pd.Categorical(heart_data['ChestPainType'], categories=['ASY', 'ATA', 'NAP', 'TA'])
heart_data['RestingECG'] = pd.Categorical(heart_data['RestingECG'], categories=['Normal', 'LVH', 'ST'])
heart_data['ST_Slope'] = pd.Categorical(heart_data['ST_Slope'], categories=['Up', 'Flat', 'Down'])


categorical_cols = ['ChestPainType', 'RestingECG', 'ST_Slope']
heart_data = pd.get_dummies(heart_data, columns=categorical_cols, drop_first=True, dtype=int)


feature_matrix = heart_data.drop("HeartDisease", axis=1)


target_labels = heart_data["HeartDisease"]


features_train, features_test, targets_train, targets_test = train_test_split(
    feature_matrix,
    target_labels,
    test_size=0.20,
    random_state=42
)

# Scaling performed to avoid features with larger ranges being seen as more significant to model
# Furthermore, it prevents data leakage by fitting to training data only.

scaler = StandardScaler()

# WWe 'fit' only on training data to prevent data leakage from the test set.
scaler.fit(features_train)

# We transform both sets using the training parameters so the scale is consistent.
features_train_scaled = scaler.transform(features_train)
features_test_scaled = scaler.transform(features_test)


In [ ]:
from sklearn.svm import SVC

# 1. Initialize the Model
# Why: 'kernel="linear"' is a solid baseline for 15 features.
# Why: 'C=1.0' is the default regularization strength.
# Why: 'random_state' ensures your results are reproducible across runs.
svm_model = SVC(kernel='linear', C=1.0, random_state=42)

# Train the Model
# Why: This uses your pre-scaled X_train to find the optimal hyperplane.
svm_model.fit(features_train_scaled, targets_train)

predictions = svm_model.predict(features_test_scaled)

